In [3]:
# NB_02_FHIR_Silver - Full SCD Type 2 Implementation
# ============================================================
# Medallion Architecture - Silver Layer
# Implements true SCD Type 2 using Delta MERGE:
#   - New records  → INSERT with is_current=True
#   - Changed records → expire old row (valid_to, is_current=False) + INSERT new
#   - Unchanged records → no action (idempotent)
# ============================================================

# === Fix for old/ancient dates ===
spark.conf.set("spark.sql.parquet.datetimeRebaseModeInWrite", "LEGACY")
spark.conf.set("spark.sql.parquet.int96RebaseModeInWrite", "LEGACY")
print("✅ Datetime rebase config applied (LEGACY mode)\n")

from pyspark.sql.functions import (
    col, to_timestamp, when, lit, current_timestamp,
    md5, concat_ws, coalesce, row_number, desc
)
from pyspark.sql.types import BooleanType, TimestampType
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import datetime


# ─────────────────────────────────────────────
# UTILITY: Safe column existence check
# ─────────────────────────────────────────────
def has_column(df, col_name: str) -> bool:
    """Returns True if col_name exists in the DataFrame schema."""
    return col_name in [field.name for field in df.schema.fields]


# ─────────────────────────────────────────────
# UTILITY: Row hash for change detection
# ─────────────────────────────────────────────
def add_row_hash(df, columns: list):
    """
    Creates md5 hash from specified columns.
    Used to detect whether a record changed between loads.
    Only hashes columns that actually exist in the DataFrame.
    """
    existing = [c for c in columns if has_column(df, c)]
    return df.withColumn(
        "row_hash",
        md5(concat_ws("||", *[coalesce(col(c).cast("string"), lit("")) for c in existing]))
    )


# ─────────────────────────────────────────────
# UTILITY: Bootstrap silver table on first run
# ─────────────────────────────────────────────
def bootstrap_silver_table(silver_df, silver_table: str):
    """
    Creates the silver Delta table from scratch on first run.
    Adds all SCD Type 2 columns before writing.
    """
    now = current_timestamp()
    init_df = silver_df \
        .withColumn("valid_from",            now) \
        .withColumn("valid_to",              lit(None).cast(TimestampType())) \
        .withColumn("is_current",            lit(True).cast(BooleanType())) \
        .withColumn("silver_load_timestamp", now)

    init_df.write.mode("overwrite").format("delta").saveAsTable(silver_table)
    count = spark.table(silver_table).count()
    print(f"   🆕 Bootstrap: created '{silver_table}' with {count:,} rows")


# ─────────────────────────────────────────────
# CORE: SCD Type 2 MERGE
# Fixed: uses .whenMatched() instead of
# .whenMatchedCondition() for Fabric compatibility
# ─────────────────────────────────────────────
def scd2_merge(silver_table: str, incoming_df, hash_columns: list):
    """
    Performs true SCD Type 2 MERGE into silver Delta table.

    Logic:
      MATCH + hash changed → expire old row + insert new row
      NO MATCH             → insert new row
      MATCH + hash same    → do nothing (idempotent)
    """
    now_ts = datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%S")

    # Add hash + SCD columns to incoming data
    incoming_hashed = add_row_hash(incoming_df, hash_columns) \
        .withColumn("valid_from",            lit(now_ts).cast(TimestampType())) \
        .withColumn("valid_to",              lit(None).cast(TimestampType())) \
        .withColumn("is_current",            lit(True).cast(BooleanType())) \
        .withColumn("silver_load_timestamp", current_timestamp())

    delta_target = DeltaTable.forName(spark, silver_table)

    # ── Step 1: Expire changed rows ──────────────────────────────
    # .whenMatched() with condition inline — works on ALL Fabric
    # Delta versions (fixes whenMatchedCondition error)
    delta_target.alias("target") \
        .merge(
            incoming_hashed.alias("source"),
            "target.id = source.id AND target.is_current = true"
        ) \
        .whenMatched("target.row_hash != source.row_hash") \
        .updateExpr({
            "valid_to":   f"'{now_ts}'",
            "is_current": "false"
        }) \
        .execute()

    # ── Step 2: Insert new + changed rows ────────────────────────
    # Re-read target after Step 1 update
    existing_current = delta_target.toDF() \
        .filter(col("is_current") == True) \
        .select("id", "row_hash")

    rows_to_insert = incoming_hashed.alias("inc") \
        .join(existing_current.alias("cur"), on="id", how="left") \
        .filter(
            col("cur.id").isNull() |                      # brand new record
            (col("inc.row_hash") != col("cur.row_hash"))  # changed record
        ) \
        .select("inc.*")

    insert_count = rows_to_insert.count()
    if insert_count > 0:
        rows_to_insert.write.format("delta").mode("append").saveAsTable(silver_table)
        print(f"   ➕ Inserted {insert_count:,} new/changed row(s)")
    else:
        print(f"   ✔️  No changes detected — silver is already up to date")


# ─────────────────────────────────────────────
# MAIN: Transform Bronze → Silver per resource
# ─────────────────────────────────────────────
def transform_to_silver(resource_type: str):
    """
    Reads bronze table, applies type casting + deduplication,
    then runs SCD Type 2 bootstrap or merge into silver table.
    """
    bronze_table = f"bronze_{resource_type.lower()}"
    silver_table = f"silver_{resource_type.lower()}"

    print(f"\n{'─'*70}")
    print(f"🔄  {bronze_table}  →  {silver_table}")
    print(f"{'─'*70}")

    raw_df = spark.table(bronze_table)

    # Deduplicate — keep latest extraction per id
    window  = Window.partitionBy("id").orderBy(desc("extraction_timestamp"))
    deduped = raw_df \
        .withColumn("_rn", row_number().over(window)) \
        .filter(col("_rn") == 1) \
        .drop("_rn") \
        .withColumn("resource_type", lit(resource_type))

    # ── Resource specific transforms + column selection ───────────

    if resource_type == "Patient":
        if has_column(deduped, "birthDate"):
            deduped = deduped.withColumn(
                "birthDate",
                when(col("birthDate").isNotNull(),
                     to_timestamp(col("birthDate"), "yyyy-MM-dd"))
            )
        silver_df = deduped.select(
            "id", "resource_type", "gender", "birthDate",
            col("name").alias("patient_name"), "address",
            "extraction_timestamp", "load_date",
            "api_base_url", "api_params"
        )
        hash_cols = ["id", "gender", "birthDate"]

    elif resource_type == "Encounter":
        if has_column(deduped, "period"):
            deduped = deduped \
                .withColumn("period_start",
                    when(col("period").isNotNull(),
                         to_timestamp(col("period").getField("start")))) \
                .withColumn("period_end",
                    when(col("period").isNotNull(),
                         to_timestamp(col("period").getField("end"))))
        silver_df = deduped.select(
            "id", "resource_type", "status", "class",
            "period_start", "period_end", "subject", "type",
            "extraction_timestamp", "load_date",
            "api_base_url", "api_params"
        )
        hash_cols = ["id", "status", "period_start", "period_end"]

    elif resource_type == "Observation":
        if has_column(deduped, "effectiveDateTime"):
            deduped = deduped.withColumn(
                "effectiveDateTime",
                when(col("effectiveDateTime").isNotNull(),
                     to_timestamp(col("effectiveDateTime")))
            )
        silver_df = deduped.select(
            "id", "resource_type", "status", "code",
            "valueQuantity", "effectiveDateTime", "subject", "encounter",
            "extraction_timestamp", "load_date",
            "api_base_url", "api_params"
        )
        hash_cols = ["id", "status", "effectiveDateTime", "valueQuantity"]

    elif resource_type == "Condition":
        for ts_col in ["onsetDateTime", "recordedDate"]:
            if has_column(deduped, ts_col):
                deduped = deduped.withColumn(
                    ts_col,
                    when(col(ts_col).isNotNull(), to_timestamp(col(ts_col)))
                )
        silver_df = deduped.select(
            "id", "resource_type", "clinicalStatus", "code",
            "subject", "encounter", "onsetDateTime", "recordedDate",
            "extraction_timestamp", "load_date",
            "api_base_url", "api_params"
        )
        hash_cols = ["id", "clinicalStatus", "onsetDateTime", "recordedDate"]

    else:
        raise ValueError(f"Unknown resource_type: {resource_type}")

    # ── Bootstrap or Merge ────────────────────────────────────────
    table_exists = spark.catalog.tableExists(silver_table)

    if not table_exists:
        print(f"   ℹ️  First run — bootstrapping '{silver_table}'...")
        bootstrap_silver_table(silver_df, silver_table)
    else:
        print(f"   ℹ️  Table exists — running SCD Type 2 merge...")
        scd2_merge(silver_table, silver_df, hash_cols)

    # ── Summary ───────────────────────────────────────────────────
    final           = spark.table(silver_table)
    total_rows      = final.count()
    current_rows    = final.filter(col("is_current") == True).count()
    historical_rows = total_rows - current_rows

    print(f"\n   📊 '{silver_table}' summary:")
    print(f"      Total rows      : {total_rows:,}")
    print(f"      Current rows    : {current_rows:,}   ← is_current = True")
    print(f"      Historical rows : {historical_rows:,}   ← is_current = False")

    display(final.filter(col("is_current") == True).limit(5))
    return silver_table


# ═══════════════════════════════════════════════════════════════════
# ENTRY POINT
# ═══════════════════════════════════════════════════════════════════
print("=" * 70)
print("🏗️   SILVER LAYER — SCD TYPE 2 TRANSFORMATION")
print("=" * 70)

results = {}
for resource in ["Patient", "Encounter", "Observation", "Condition"]:
    try:
        transform_to_silver(resource)
        results[resource] = "✅ SUCCESS"
    except Exception as e:
        results[resource] = f"❌ FAILED: {e}"
        print(f"\n❌ Error on {resource}: {e}")

# ── Run Summary ───────────────────────────────────────────────────
print("\n" + "=" * 70)
print("📋  SILVER LAYER RUN SUMMARY")
print("=" * 70)
for resource, status in results.items():
    print(f"   {resource:<15} → {status}")


# ═══════════════════════════════════════════════════════════════════
# VERIFICATION — SCD Type 2 status per table
# ═══════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("🔍  SCD TYPE 2 VERIFICATION")
print("=" * 70)

for resource in ["Patient", "Encounter", "Observation", "Condition"]:
    silver_table = f"silver_{resource.lower()}"
    try:
        df         = spark.table(silver_table)
        total      = df.count()
        current    = df.filter(col("is_current") == True).count()
        historical = total - current

        print(f"\n   {silver_table}")
        print(f"      Total      : {total:,}")
        print(f"      Current    : {current:,}")
        print(f"      Historical : {historical:,}")

        if historical > 0:
            print(f"      🔄 SCD Type 2 history EXISTS — changes were tracked!")
        else:
            print(f"      ✔️  No history yet — expected on first/unchanged run")
    except Exception as e:
        print(f"   ❌ {silver_table}: {e}")

StatementMeta(, 75ff9b00-eb4d-49cc-8e8d-ba6dbe55bcb8, 5, Finished, Available, Finished, False)

✅ Datetime rebase config applied (LEGACY mode)

🏗️   SILVER LAYER — SCD TYPE 2 TRANSFORMATION

──────────────────────────────────────────────────────────────────────
🔄  bronze_patient  →  silver_patient
──────────────────────────────────────────────────────────────────────
   ℹ️  Table exists — running SCD Type 2 merge...

❌ Error on Patient: 'DeltaMergeBuilder' object has no attribute 'whenMatched'

──────────────────────────────────────────────────────────────────────
🔄  bronze_encounter  →  silver_encounter
──────────────────────────────────────────────────────────────────────
   ℹ️  Table exists — running SCD Type 2 merge...

❌ Error on Encounter: 'DeltaMergeBuilder' object has no attribute 'whenMatched'

──────────────────────────────────────────────────────────────────────
🔄  bronze_observation  →  silver_observation
──────────────────────────────────────────────────────────────────────
   ℹ️  Table exists — running SCD Type 2 merge...

❌ Error on Observation: 'DeltaMergeBuilder'